# Module 11 — Dataclasses, Enums, and Value Semantics

## Exercise 11.3 — Replace stringly-typed state with enums

The order-processing code below uses strings for status, permissions and
priority. Every bug in it is a bug that an enum makes impossible.
Run:  python ex03_enums.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. `@dataclass`

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float
    label: str = ""

That generates `__init__`, `__repr__`, and `__eq__`. The annotations are not
decoration — the decorator reads `__annotations__` at class creation time to
find the fields, which is why a field without an annotation is silently ignored:

In [ ]:
@dataclass
class Broken:
    x: int
    y = 0            # NO annotation -> a class attribute, NOT a field
                     # It will not appear in __init__, __repr__ or __eq__.

### The options that matter

```text
@dataclass(
    frozen=True,      # immutable: __setattr__ raises; also generates __hash__
    slots=True,       # 3.10+: generate __slots__, saving ~40% memory
    order=True,       # generate __lt__, __le__, __gt__, __ge__ from field order
    kw_only=True,     # 3.10+: all fields keyword-only at the call site
    eq=True,          # default; set False to keep identity comparison
    repr=True,        # default
)
```


**`frozen=True` should be your default.** It gives you `__hash__` for free,
makes aliasing bugs (Module 02) unrepresentable, and makes the object safe to
share between threads (Module 21) and to use as a dict key or cache key.

**`slots=True` costs nothing** for a data-shaped class and saves real memory
(Module 08). The exception is anything needing `cached_property` or `weakref`.

**`kw_only=True` for anything with more than three fields.**
`Config(30, 3, True, False)` is unreadable; `Config(timeout=30, retries=3, ...)`
is not.

**`order=True` compares fields in declaration order**, as a tuple. If that is
not the ordering you want, write `__lt__` yourself — silently sorting by the
wrong field is worse than not sorting.

### `field()`

In [ ]:
@dataclass
class Config:
    name: str
    tags: list[str] = field(default_factory=list)      # NOT `= []`
    _cache: dict = field(default_factory=dict, repr=False, compare=False)
    created: datetime = field(default_factory=datetime.now)
    version: int = field(default=1, metadata={"docs": "schema version"})

`default_factory` runs **per instance**, which is the fix for Module 02's
mutable-default trap. `@dataclass` refuses a mutable default outright:

In [ ]:
@dataclass
class Bad:
    items: list = []      # ValueError: mutable default <class 'list'> ...

That is one of the nicest things about dataclasses: a whole bug category becomes
a startup error.

`compare=False` excludes a field from `__eq__` and ordering — right for caches,
timestamps, and derived values. `repr=False` keeps secrets out of logs
(Module 08).

### `__post_init__`

In [ ]:
@dataclass(frozen=True)
class Rectangle:
    width: float
    height: float
    area: float = field(init=False)        # computed, not passed in

    def __post_init__(self) -> None:
        if self.width <= 0 or self.height <= 0:
            raise ValueError(f"dimensions must be positive: {self}")
        object.__setattr__(self, "area", self.width * self.height)
        # object.__setattr__ because frozen blocks the normal assignment.
        # This is the documented way to set computed fields on a frozen
        # dataclass, and it is the one place you should use it.

Validation in `__post_init__` means **an invalid instance cannot exist**
(Module 08's rule), and every caller — the constructor, the deserializer, the
test fixture — gets it.

### Working with instances

In [ ]:
from dataclasses import replace, asdict, astuple, fields

p2 = replace(p, x=10)          # a NEW instance with one field changed
asdict(p)                       # recursive dict; follows nested dataclasses
astuple(p)                      # recursive tuple
[f.name for f in fields(p)]     # introspection

`replace()` is how you "modify" a frozen dataclass, and it is the pattern that
makes immutability practical.

Note that `asdict()` **deep-copies** everything, including nested dataclasses,
lists and dicts. That is usually what you want and occasionally an expensive
surprise.

---

## Concept 3. `Enum`

In [ ]:
from enum import Enum, IntEnum, StrEnum, auto, Flag

class Status(Enum):
    PENDING = "pending"
    ACTIVE = "active"
    CLOSED = "closed"

Status.ACTIVE           # <Status.ACTIVE: 'active'>
Status.ACTIVE.value     # 'active'
Status("active")        # lookup BY VALUE -> Status.ACTIVE
Status["ACTIVE"]        # lookup by NAME
list(Status)            # iterable, in definition order

Enums replace magic strings and give you three things a string cannot: a typo is
a `ValueError` at the boundary rather than a silent no-match, the valid set is
discoverable and iterable, and a type checker can verify exhaustiveness in a
`match`.

In [ ]:
class Priority(IntEnum):        # comparable and usable as an int
    LOW = 1
    HIGH = 3

Priority.HIGH > Priority.LOW    # True
sorted(tasks, key=lambda t: t.priority)

class Colour(StrEnum):          # 3.11+: IS a str, so it JSON-serialises
    RED = "red"

json.dumps({"c": Colour.RED})   # works; a plain Enum raises

class Perm(Flag):               # combinable
    READ = auto()
    WRITE = auto()
    ALL = READ | WRITE

Perm.READ in (Perm.READ | Perm.WRITE)     # True

`IntEnum` and `StrEnum` exist for interoperability with code that expects a
plain int or str — serialization, database columns, HTTP headers. Prefer plain
`Enum` unless you need that, because the looseness that makes them convenient
also lets `Status.ACTIVE == "active"` be True, which defeats part of the point.

Enums with behaviour are fine and underused:

In [ ]:
class Status(Enum):
    PENDING = "pending"
    ACTIVE = "active"

    @property
    def is_terminal(self) -> bool:
        return self is Status.CLOSED

    @classmethod
    def from_legacy_code(cls, code: int) -> "Status":
        return {0: cls.PENDING, 1: cls.ACTIVE}[code]

---

## Concept 4. Value semantics

A **value object** is defined by its contents, not its identity. Two `Money`
objects holding $5 are interchangeable; two `BankAccount` objects with a $5
balance are not.

| | Value object | Entity |
|---|---|---|
| Identity | its contents | an ID that outlives changes |
| Equality | field by field | by ID |
| Mutability | immutable | usually mutable |
| Examples | `Money`, `Point`, `DateRange`, `Email` | `User`, `Order`, `Account` |
| Build with | `@dataclass(frozen=True)` | `@dataclass` with an id field |

**Prefer value objects.** The benefits compound:

- Aliasing bugs cannot happen (Module 02).
- Hashable, so usable as dict keys and cache keys.
- Thread-safe for free (Module 21) — no lock can be forgotten if there is
  nothing to protect.
- Trivially testable: construct, assert, done. No setup, no teardown.
- Easy to reason about: a value that cannot change cannot change *behind you*.

The objection is allocation cost. For records at ordinary scale it does not
matter; measure before you let it drive the design (Module 23).

### Making illegal states unrepresentable

In [ ]:
# weak: every consumer must re-check
@dataclass
class Order:
    status: str
    shipped_at: datetime | None = None

# strong: the type enforces it
@dataclass(frozen=True)
class Pending: ...

@dataclass(frozen=True)
class Shipped:
    shipped_at: datetime          # cannot be absent

Order = Pending | Shipped

In the second version, "shipped with no timestamp" cannot be constructed, so no
code needs to handle it and no test needs to cover it. Combined with `match`
(Module 04), the type checker verifies you handled every case.

This is the highest-leverage idea in Part 2: **push invariants into types, so
that the checking happens once at construction rather than everywhere else
forever.**

---

## Concept 5. Copy semantics revisited

In [ ]:
import copy

@dataclass
class Team:
    name: str
    members: list[str]

a = Team("eng", ["ada"])
b = copy.copy(a)                # shallow: SAME list
b.members.append("bo")
a.members                        # ['ada', 'bo']   <-- Module 02 again

c = copy.deepcopy(a)            # independent
d = replace(a, name="ops")      # NEW object, but members is still SHARED

**`replace()` is a shallow copy.** It creates a new instance with the fields you
name changed and the rest **shared**. For a fully frozen structure that is
perfect and free. For one containing a mutable field, it is the shallow-copy
trap wearing a dataclass costume.

The fix is not to remember it — the fix is to make the fields immutable:

In [ ]:
@dataclass(frozen=True)
class Team:
    name: str
    members: tuple[str, ...] = ()      # tuple, not list

Now `replace()` is always safe, because nothing reachable can change. Note that
`frozen=True` alone would **not** have saved you: it prevents rebinding the
attribute, not mutating the list it points at. Module 02's tuple trap, one more
time, and it is the single most common dataclass mistake.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `@dataclass`
- Section 2: Choosing a record type
- Section 3: `Enum`
- Section 4: Value semantics
- Section 5: Copy semantics revisited

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime

---

## `Order`

_Order_

In [ ]:
@dataclass
class Order:
    id: int
    status: str = "pending"
    priority: str = "normal"
    permissions: list[str] = field(default_factory=list)
    history: list[tuple[datetime, str]] = field(default_factory=list)

---

## `transition`

BUG 1: a typo in new_status raises KeyError far from its cause -- or

In [ ]:
def transition(order: Order, new_status: str) -> None:
    """BUG 1: a typo in new_status raises KeyError far from its cause -- or
    worse, silently passes if the typo is in VALID_TRANSITIONS itself."""
    if new_status not in VALID_TRANSITIONS[order.status]:
        raise ValueError(f"cannot go from {order.status} to {new_status}")
    order.status = new_status
    order.history.append((datetime.now(), new_status))

---

## `can_edit`

BUG 2: permission strings are compared by value, so 'Admin' and 'admin'

In [ ]:
def can_edit(order: Order, user_permissions: list[str]) -> bool:
    """BUG 2: permission strings are compared by value, so 'Admin' and 'admin'
    are different permissions, and nothing enumerates the valid set."""
    return "admin" in user_permissions or (
        "editor" in user_permissions and order.status == "pending"
    )

---

## `priority_score`

BUG 3: an unknown priority silently scores 0, so a typo'd 'urgnet' order

In [ ]:
def priority_score(order: Order) -> int:
    """BUG 3: an unknown priority silently scores 0, so a typo'd 'urgnet' order
    sorts LAST rather than first. Nothing fails; the wrong thing just happens."""
    return {"low": 1, "normal": 2, "high": 3, "urgent": 4}.get(order.priority, 0)

---

## `describe`

BUG 4: adding a new status requires finding and updating every one of

In [ ]:
def describe(order: Order) -> str:
    """BUG 4: adding a new status requires finding and updating every one of
    these chains, and nothing tells you when you have missed one."""
    if order.status == "pending":
        return "awaiting payment"
    elif order.status == "paid":
        return "preparing"
    elif order.status == "shipped":
        return "in transit"
    else:
        return "finished"

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    o = Order(1)
    assert o.status is Status.PENDING              # type: ignore[name-defined]

    transition(o, Status.PAID)                     # type: ignore[name-defined]
    assert o.status is Status.PAID                 # type: ignore[name-defined]

    try:
        transition(o, Status.DELIVERED)            # type: ignore[name-defined]
    except ValueError:
        pass
    else:
        raise AssertionError("invalid transition must raise")

    perms = Permission.READ | Permission.WRITE     # type: ignore[name-defined]
    assert Permission.WRITE in perms               # type: ignore[name-defined]
    assert Permission.ADMIN not in perms           # type: ignore[name-defined]

    assert Priority.URGENT > Priority.LOW          # type: ignore[name-defined]
    orders = [Order(1, priority=Priority.LOW),     # type: ignore[name-defined]
              Order(2, priority=Priority.URGENT)]  # type: ignore[name-defined]
    assert sorted(orders, key=lambda x: x.priority, reverse=True)[0].id == 2

    assert Status.parse("paid") is Status.PAID     # type: ignore[name-defined]
    try:
        Status.parse("payed")                      # type: ignore[name-defined]
    except ValueError as exc:
        assert "paid" in str(exc), "the error must list the valid values"
    else:
        raise AssertionError("a typo must be rejected at the boundary")

    print("all enum checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.